### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))
backtesting_transaction_costs = pd.read_parquet(cfg.data_path("backtesting_costs"))
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
import numpy as np

# --- Evaluation aus src/ ---
sys.path.insert(0, "..")
from src.backtest.evaluation import evaluate_strategies

# Daten laden
test_df = pd.read_parquet(cfg.data_path("test_data"))
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))

# --- DYNAMISCHE ZUORDNUNG DER SIGNALE ---
signals_to_count = pd.DataFrame(index=test_df.index)
signal_columns = [c for c in test_df.columns if c.endswith('_Signal')]

for sig_col in signal_columns:
    model_name = sig_col.rsplit('_', 1)[0]
    signals_to_count[model_name] = test_df[sig_col]

# Evaluation ausführen
evaluation_table = evaluate_strategies(backtesting_results, signals_to_count, backtesting_transaction_costs)

# Anzeige und Persistierung
print("\n--- UMFASSENDE EVALUATION (DYNAMISCH) ---")
display(evaluation_table)

evaluation_table.to_markdown(cfg.asset_path("evaluation_table"), index=True)
print(f"\nEvaluationstabelle erfolgreich unter {cfg.asset_path('evaluation_table')} persistiert.")

In [ ]:
# --- 06_MCS: Block-Bootstrap Robustness-Check ---

import matplotlib.pyplot as plt
import os

# --- MCS aus src/ ---
from src.backtest.evaluation import run_monte_carlo_simulation
from src.backtest.sorr import build_sorr_scenarios

# Parameter aus zentraler Config
N_SIMULATIONS = cfg.evaluation.mcs.n_paths
BLOCK_SIZE = cfg.evaluation.mcs.block_length
RANDOM_SEED = cfg.evaluation.mcs.random_seed
SIM_YEARS = cfg.evaluation.mcs.sim_years
TRADING_DAYS = cfg.evaluation.mcs.trading_days_per_year
TOTAL_DAYS = SIM_YEARS * TRADING_DAYS

# Definition der Szenarien aus zentraler Config
scenarios = build_sorr_scenarios(cfg.backtesting.sorr.scenarios)

# Tägliche Netto-Renditen der Strategien berechnen
daily_rets = backtesting_results.pct_change().dropna()

# Monte Carlo Simulation ausführen
all_mc_summaries, mcs_paths_collector = run_monte_carlo_simulation(
    daily_rets=daily_rets,
    test_df=test_df,
    scenarios=scenarios,
    n_simulations=N_SIMULATIONS,
    block_size=BLOCK_SIZE,
    random_seed=RANDOM_SEED,
    sim_years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS,
)

# --- Boxplot-Visualisierung pro Szenario ---
# (MCS liefert final_capitals nicht direkt zurück, daher aus mcs_paths_collector ableiten)
for sc_name, params in scenarios.items():
    mc_results_scenario = {}
    for strategy in daily_rets.columns:
        prefix = f"{sc_name}_{strategy}_path_"
        strat_paths = {k: v for k, v in mcs_paths_collector.items() if k.startswith(prefix)}
        if strat_paths:
            mc_results_scenario[strategy] = [path[-1] for path in strat_paths.values()]

    if mc_results_scenario:
        plt.figure(figsize=(12, 6))
        plt.boxplot(mc_results_scenario.values(),
                    tick_labels=[s.replace('_', ' ') for s in mc_results_scenario.keys()])
        plt.title(f"MCS {sc_name}: Verteilung des Endkapitals (Start: {params['start']:,.0f}€)")
        plt.ylabel(f"Endkapital nach {SIM_YEARS} Jahren in €")
        plt.axhline(y=0, color='red', linestyle='--')
        plt.grid(axis='y', alpha=0.3)
        plt.savefig(cfg.asset_path(f"mcs_boxplot_{sc_name.lower()}"), dpi=300, bbox_inches='tight')
        plt.show()

# Ergebnisse zusammenführen und als Markdown-Tabelle speichern
if all_mc_summaries:
    mc_multi_summary = pd.DataFrame(all_mc_summaries).set_index(['Szenario', 'Strategie'])
    mc_multi_summary.to_markdown(cfg.asset_path("mcs_summary"))
    display(mc_multi_summary)

print("Alle Monte Carlo Simulationen abgeschlossen.")

mcs_results = pd.DataFrame(mcs_paths_collector)
print(mcs_results)

# ---

# Liste der Szenarien und Strategien dynamisch aus Config extrahieren
scenarios_list = list(vars(cfg.backtesting.sorr.scenarios).keys())
strategies = backtesting_results.columns

color_map = cfg.color_map
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# --- Plot: MCS Pfad-Verläufe ---
fig, axes = plt.subplots(len(scenarios_list), 1, figsize=(15, 6 * len(scenarios_list)), sharex=True)

for ax, sc_name in zip(axes, scenarios_list):
    print(f"Plotte Pfad-Verläufe für {sc_name}...")

    for strat in strategies:
        prefix = f"{sc_name}_{strat}_path_"
        strat_paths = mcs_results.filter(like=prefix)

        color = color_map.get(strat, 'black')
        alpha = 0.05

        ax.plot(strat_paths, color=color, alpha=alpha, linewidth=1)
        ax.plot([], [], color=color, label=strat.replace('_', ' '))

    ax.set_title(f"MCS Pfad-Verläufe: Szenario {sc_name}")
    ax.set_ylabel("Kapital in €")
    ax.axhline(y=0, color='black', linewidth=1.5)
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left', ncol=2)

plt.xlabel("Handelstage (T+N)")
plt.tight_layout()
plt.savefig(cfg.asset_path("mcs_paths"), dpi=300, bbox_inches='tight')
plt.show()

# --- Plot: MCS Quantils-Verteilung ---
fig, axes = plt.subplots(len(scenarios_list), 1, figsize=(15, 6 * len(scenarios_list)), sharex=True)

for ax, sc_name in zip(axes, scenarios_list):
    print(f"Berechne Quantile für {sc_name}...")

    for strat in strategies:
        prefix = f"{sc_name}_{strat}_path_"
        strat_paths = mcs_results.filter(like=prefix)

        upper_95 = strat_paths.quantile(0.95, axis=1)
        lower_05 = strat_paths.quantile(0.05, axis=1)
        median_50 = strat_paths.median(axis=1)

        color = color_map.get(strat, 'black')

        ax.fill_between(range(TOTAL_DAYS), lower_05, upper_95, color=color, alpha=0.15)
        ax.plot(median_50, color=color, linewidth=1.5, label=f"{strat.replace('_', ' ')} (Median)")

    ax.set_title(f"MCS Konfidenz-Intervalle (5% - 95%): Szenario {sc_name}")
    ax.set_ylabel("Kapital in €")
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label="Erschöpfungsgrenze")
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left', ncol=2)

plt.xlabel("Handelstage (T+N)")
plt.tight_layout()
plt.savefig(cfg.asset_path("mcs_quantiles"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
output_path = cfg.data_path('mcs_data')

# Speichern als Parquet
mcs_results.to_parquet(output_path)

print(f"MCS-Dataframe erfolgreich unter {output_path} gespeichert.")